In [20]:
import numpy as np
import MDAnalysis as mda
from MDAnalysis import Universe
from MDAnalysis.tests.datafiles import PSF

u = mda.Universe("4BS2+POPC.gro","4BS2+POPC_PBC_MOL_CENTER.xtc")


lipids = u.select_atoms("resname POPC")

ref_state = u.trajectory[0]

ref_x = ref_state.dimensions[0]
ref_y = ref_state.dimensions[1]

stresses = np.linspace(-100, 100, 21) 
APLs = []
for stress in stresses:
    box = ref_state.dimensions
    box[0] += stress / 100  
    box[1] -= stress / 100
    u.dimensions = box
    # Calculate the area per lipid
    area = u.trajectory[-1].volume / len(lipids)
    APLs.append(area)
dAPL = np.array(APLs) - APLs[10]
stress = stresses * 1e5 
modulus = np.polyfit(stress, dAPL, 1)[0] / APLs[10]

print(f"Young's modulus: {modulus:.2f} Pa")


Young's modulus: 0.00 Pa


In [44]:
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
from __future__ import print_function
%matplotlib inline
import numpy as np
import mdtraj as md

u = mda.Universe("4BS2+POPC.gro","4BS2+POPC_PBC_MOL_CENTER.xtc")
lipids = u.select_atoms("resname POPC")

box_dimensions = u.dimensions[:3]

num_lipids = len(lipids) //134
print(num_lipids)


trajectory = md.load('4BS2+POPC.gro' , '4BS2+POPC_PBC_MOL_CENTER.xtc')
sasa = md.shrake_rupley(trajectory)

total_sasa = sasa.sum(axis=1)

#sasa = SASA(lipids, mode='residue')
#sasa.run()
#total_area = SASA(lipids).run().sasa.sum()

700


In [45]:
total_sasa = sasa.sum(axis=1)
print(total_sasa)

[3173.2078]


In [47]:
apl = total_sasa / num_lipids
print(apl)

[4.533154]


In [50]:
surface_tensions = np.arange(0, 80, 5)
apl_data = []
for tension in surface_tensions:
    u.dimensions[:2] += [0, 0, tension/100]
    box_dimensions = u.dimensions[:3]
    
    num_lipids = len(lipids) // 134
    apl = total_area / num_lipids
    apl_data.append(apl)
    
    u.dimensions[:2] -= [0, 0, tension/100]
surface_tensions = np.array(surface_tensions)
apl_data = np.array(apl_data)
regression = np.polyfit(surface_tensions, apl_data, 1)
slope = regression[0]
youngs_modulus = slope * apl / 2
print("Young's modulus:", youngs_modulus, "Pa")    

ValueError: operands could not be broadcast together with shapes (2,) (3,) (2,) 

[151.841 151.841 141.068  90.     90.     90.   ]
